# HANAS Phase 2 Results and Discussion

The Phase 2 experiment compares the rule-based baseline controller and the Agentic AI controller for automated pH and electrical conductivity regulation in a hydroponic reservoir.

The analysis follows the seven-day experimental protocol:

| Day | Scenario | Expected behavior |
|---:|---|---|
| 1 | Stable pH and EC | No dosing from either controller |
| 2 | pH low, EC stable | pH Up correction |
| 3 | pH high, EC stable | pH Down correction |
| 4 | EC low, pH stable | EC Up A/B correction |
| 5-6 | EC high, pH stable | EC Down dilution and longer recovery |
| 7 | Mixed pH/EC disturbance | Bounded correction, remeasurement, and follow-up correction when needed |

After Days 2-7 corrections, at least ten confirmation readings are used before ending a trial. The notebook reports a stricter dose-level estimate: after each dose first returns the affected parameter to range, the next ten logged readings must remain in range. If drift occurs, a later re-dose starts a new response-and-confirmation episode.


## 1. Benchmarking Framework

For this control-system experiment, the best metrics are not standard machine-learning metrics such as accuracy, precision, or recall. The controller is not classifying images. It is maintaining a chemical process within target bounds while minimizing unnecessary actuation.

### Primary Metrics

1. **Combined pH-and-EC in-range percentage**: strongest overall water-quality metric.
2. **pH in-range percentage**: acid/base control quality.
3. **EC in-range percentage**: nutrient-strength and dilution control quality.
4. **Dosing events per 100 readings**: normalized actuation burden.
5. **Immediate ten-reading post-return stability success**: dose-level estimate of whether the ten logged readings after first return-to-range remained in range.

### Secondary Metrics

1. Mean absolute deviation from target range.
2. 95th percentile absolute deviation.
3. Median and mean dose-response time.
4. Pump-specific dose count, dose volume, and pump runtime.
5. Detection-to-action latency and actuation duration.
6. Sensor anomaly and wait-state frequency.

### Statistical Positioning

The exported readings are time-series operational logs, so adjacent readings are not fully independent. This notebook therefore reports:

- descriptive performance metrics,
- confidence intervals,
- two-proportion tests as supporting evidence,
- effect sizes using Cohen's h,
- cluster bootstrap intervals by experiment run to reduce overconfidence from autocorrelation.


In [ ]:
from pathlib import Path
from math import asin, sqrt, erfc
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# Robust path handling whether the notebook is launched from the repo root,
# research_analysis/, or research_analysis/notebooks/.
def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_repo_markers = (candidate / "hanas_backend").exists() and (candidate / "research_analysis" / "requirements.txt").exists()
        has_data_dir = (candidate / "research_analysis").exists()
        if has_repo_markers and has_data_dir:
            return candidate
    raise FileNotFoundError("Could not locate the HANAS project root from the current notebook directory.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "research_analysis"
RAW_DATA_DIR = DATA_DIR / "raw_data"

WORKBOOK_PATH = RAW_DATA_DIR / "phase2" / "Phase 2 Raw Result - Extended.xlsx"
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Could not find the Phase 2 workbook: {WORKBOOK_PATH}")

REPORT_DIR = DATA_DIR / "reports" / "phase2_figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTICLE_FIGURE_DIR = DATA_DIR / "article" / "1-MDPI" / "figures" / "phase2"
ARTICLE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PH_MIN = 5.5
TARGET_PH_MAX = 6.5
TARGET_EC_MIN = 1.2
TARGET_EC_MAX = 2.0

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
ORDER = ["baseline", "agentic_ai"]
CONTROLLER_LABELS = {"baseline": "Baseline", "agentic_ai": "Agentic AI"}
DISPLAY_ORDER = [CONTROLLER_LABELS[item] for item in ORDER]
DISPLAY_PALETTE = {"Baseline": "#5B7FA3", "Agentic AI": "#D9852F"}
PALETTE = {raw: DISPLAY_PALETTE[label] for raw, label in CONTROLLER_LABELS.items()}

SCENARIO_ORDER = ["stable", "ph_low", "ph_high", "ec_low", "ec_high", "mixed"]
SCENARIO_LABELS = {
    "stable": "Stable",
    "ph_low": "pH Low",
    "ph_high": "pH High",
    "ec_low": "EC Low",
    "ec_high": "EC High",
    "mixed": "Mixed",
}
SCENARIO_DISPLAY_ORDER = [SCENARIO_LABELS[item] for item in SCENARIO_ORDER]

PUMP_LABELS = {
    "ec_down": "EC Down",
    "ec_up": "EC Up",
    "ph_down": "pH Down",
    "ph_up": "pH Up",
}

DECISION_LABELS = {
    "within_range": "Within Range",
    "wait_initial_confirmation": "Wait: Initial Confirmation",
    "wait_mixing": "Wait: Mixing",
    "wait_stability": "Wait: Stability",
    "sensor_anomaly": "Sensor Anomaly",
    "ec_high": "EC High",
    "ec_low": "EC Low",
    "ph_high": "pH High",
    "ph_low": "pH Low",
}

ARTICLE_FIGURE_NAMES = {
    "fig1_range_compliance.png": "range_compliance.png",
    "fig2_dosing_burden.png": "dosing_burden.png",
    "fig3_scenario_compliance.png": "scenario_compliance.png",
    "fig4_scenario_dosing_burden.png": "scenario_dosing_burden.png",
    "fig5_pump_events_and_volume.png": "pump_events_and_volume.png",
    "fig6_response_and_stability.png": "response_and_stability.png",
    "fig7_timeseries_with_dosing.png": "timeseries_with_dosing.png",
    "fig8_decision_distribution.png": "decision_distribution.png",
}

def display_controller(value):
    return CONTROLLER_LABELS.get(str(value), str(value).replace("_", " ").title())

def display_scenario(value):
    return SCENARIO_LABELS.get(str(value), str(value).replace("_", " ").title())

def display_decision(value):
    return DECISION_LABELS.get(str(value), str(value).replace("_", " ").title())

def display_pump(value):
    return PUMP_LABELS.get(str(value), str(value).replace("_", " ").title())

def save_figure(fig, report_filename):
    report_path = REPORT_DIR / report_filename
    fig.savefig(report_path, dpi=300, bbox_inches="tight")
    article_name = ARTICLE_FIGURE_NAMES.get(report_filename)
    if article_name:
        shutil.copy2(report_path, ARTICLE_FIGURE_DIR / article_name)
    return report_path

print(f"Project root: {PROJECT_ROOT}")
print(f"Workbook: {WORKBOOK_PATH}")
print(f"Figure output directory: {REPORT_DIR}")
print(f"Article figure output directory: {ARTICLE_FIGURE_DIR}")


In [ ]:
# Load every worksheet from the Phase 2 export.
sheets = pd.read_excel(WORKBOOK_PATH, sheet_name=None, engine="openpyxl")

sheet_overview = pd.DataFrame([
    {"worksheet": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in sheets.items()
])
display(sheet_overview)

## 2. Data Preparation and Validation

The Phase 2 workbook contains `experiment_runs`, `control_cycles`, `system_logs`, `trial_events`, and `reference_ranges`. The primary benchmarking table is `system_logs`, while `control_cycles` provides timing support. The `trial_events` worksheet links annotated disturbance events to `system_logs` through `system_logs_id`, enabling severity-aware scenario interpretation.

Scenario and severity labels are assigned from `trial_events` when available. If an analysis workbook does not include event annotations, the notebook falls back to run-level scenario inference from controller correction decisions.


In [ ]:
experiment_runs = sheets["experiment_runs"].copy()
system_logs = sheets["system_logs"].copy()
control_cycles = sheets["control_cycles"].copy()
reference_ranges = sheets["reference_ranges"].copy()
trial_events = sheets.get("trial_events", pd.DataFrame()).copy()

# Normalize timestamps.
for df, cols in [
    (experiment_runs, ["start_time", "end_time"]),
    (system_logs, ["timestamp"]),
    (control_cycles, ["started_at", "deviation_detected_at", "action_started_at", "action_completed_at", "completed_at"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Enrich system logs.
for col in ["ph", "ec", "temperature", "reservoir_volume_liters", "dose_ml", "duration_ms", "ph_deviation", "ec_deviation"]:
    system_logs[col] = pd.to_numeric(system_logs[col], errors="coerce")

if not trial_events.empty:
    for col in ["experiment_run_id", "system_logs_id", "trial_number", "working_volume_liters"]:
        if col in trial_events.columns:
            trial_events[col] = pd.to_numeric(trial_events[col], errors="coerce")
    for col in ["event_type", "disturbance_type", "target_metric", "controller_type", "severity_level", "disturbed_range", "episode_name"]:
        if col in trial_events.columns:
            trial_events[col] = trial_events[col].fillna("").astype(str).str.strip()

def coerce_bool_series(series):
    """Parse exported booleans consistently when Excel loads them as strings."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int).astype(bool)
    normalized = series.fillna("").astype(str).str.strip().str.lower()
    return normalized.map({
        "true": True, "1": True, "yes": True,
        "false": False, "0": False, "no": False,
        "none": False, "nan": False, "": False,
    }).fillna(False)

exported_ph_within = coerce_bool_series(system_logs["ph_within_range"])
exported_ec_within = coerce_bool_series(system_logs["ec_within_range"])
computed_ph_within = system_logs["ph"].between(TARGET_PH_MIN, TARGET_PH_MAX, inclusive="both")
computed_ec_within = system_logs["ec"].between(TARGET_EC_MIN, TARGET_EC_MAX, inclusive="both")

system_logs["ph_within_range"] = computed_ph_within
system_logs["ec_within_range"] = computed_ec_within
system_logs["both_within_range"] = system_logs["ph_within_range"] & system_logs["ec_within_range"]

system_logs["ph_deviation"] = np.select(
    [system_logs["ph"] < TARGET_PH_MIN, system_logs["ph"] > TARGET_PH_MAX],
    [TARGET_PH_MIN - system_logs["ph"], system_logs["ph"] - TARGET_PH_MAX],
    default=0,
)
system_logs["ec_deviation"] = np.select(
    [system_logs["ec"] < TARGET_EC_MIN, system_logs["ec"] > TARGET_EC_MAX],
    [TARGET_EC_MIN - system_logs["ec"], system_logs["ec"] - TARGET_EC_MAX],
    default=0,
)

pump_label = system_logs["pump_activated"].fillna("none").astype(str).str.strip().str.lower()
system_logs["pump_activated_norm"] = pump_label
system_logs["is_dose_event"] = (~system_logs["pump_activated_norm"].isin(["none", "nan", ""])) & (
    (system_logs["duration_ms"].fillna(0) > 0) | (system_logs["dose_ml"].fillna(0) > 0)
)
system_logs["date"] = system_logs["timestamp"].dt.date
system_logs["hour"] = system_logs["timestamp"].dt.hour

def parse_metadata(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value) or value == "":
        return {}
    try:
        return json.loads(value)
    except Exception:
        return {}

system_logs["metadata"] = system_logs["decision_metadata"].apply(parse_metadata)

print("System log date range:", system_logs["timestamp"].min(), "to", system_logs["timestamp"].max())
print("Control strategies:", system_logs["control_strategy"].dropna().unique())
print("Raw dose events:", int(system_logs["is_dose_event"].sum()))
print("Trial events:", len(trial_events))

validation_rows = []
validation_rows.append({"check": "pH exported flag mismatches", "count": int((exported_ph_within != computed_ph_within).sum())})
validation_rows.append({"check": "EC exported flag mismatches", "count": int((exported_ec_within != computed_ec_within).sum())})
validation_rows.append({"check": "missing timestamps", "count": int(system_logs["timestamp"].isna().sum())})
validation_rows.append({"check": "missing pH values", "count": int(system_logs["ph"].isna().sum())})
validation_rows.append({"check": "missing EC values", "count": int(system_logs["ec"].isna().sum())})
if not trial_events.empty:
    missing_link_count = int((~trial_events["system_logs_id"].isin(system_logs["id"])).sum())
    validation_rows.append({"check": "trial events without matching system log", "count": missing_link_count})

duplicate_signature = [
    "experiment_run_id", "control_strategy", "timestamp", "ph", "ec",
    "temperature", "reservoir_volume_liters", "decision", "pump_activated",
    "dose_ml", "duration_ms", "status",
]
duplicate_mask = system_logs.duplicated(duplicate_signature, keep=False)
duplicate_drop_mask = system_logs.duplicated(duplicate_signature, keep="first")
validation_rows.append({"check": "repeated system-log signature rows", "count": int(duplicate_mask.sum())})
validation_rows.append({"check": "repeated rows excluded from metrics", "count": int(duplicate_drop_mask.sum())})

validation_summary = pd.DataFrame(validation_rows)
display(validation_summary)
if duplicate_mask.any():
    display(system_logs.loc[duplicate_mask, ["id", "control_cycle_id", *duplicate_signature]].sort_values(duplicate_signature).head(30))

# Exclude repeated measurement signatures from metric computation while retaining
# the validation output above for traceability.
if duplicate_drop_mask.any():
    print(f"Excluded repeated system-log measurement rows: {int(duplicate_drop_mask.sum())}")
    system_logs = system_logs.loc[~duplicate_drop_mask].copy()


In [ ]:
def infer_scenario_from_decisions(decisions):
    correction_labels = ["ph_low", "ph_high", "ec_low", "ec_high"]
    present = [label for label in correction_labels if (decisions == label).any()]
    if len(present) == 0:
        return "stable"
    if len(present) > 1:
        return "mixed"
    return present[0]

def infer_scenario_from_trial_events(events):
    if events.empty:
        return {}
    scenario_map = {}
    for run_id, group in events.groupby("experiment_run_id"):
        disturbance_types = sorted({str(v).strip() for v in group["disturbance_type"] if str(v).strip()})
        if len(disturbance_types) == 0:
            scenario_map[run_id] = "stable"
        elif len(disturbance_types) == 1:
            scenario_map[run_id] = disturbance_types[0]
        else:
            scenario_map[run_id] = "mixed"
    return scenario_map

def severity_rank_value(severity):
    return {"": 0, "mild": 1, "moderate": 2, "severe": 3, "extreme": 4}.get(str(severity).strip(), 0)

def highest_severity(values):
    cleaned = [str(v).strip() for v in values if str(v).strip()]
    if not cleaned:
        return ""
    return max(cleaned, key=severity_rank_value)

def joined_unique(values):
    cleaned = sorted({str(v).strip() for v in values if str(v).strip()})
    return "+".join(cleaned)

def assign_disturbance_episode_annotations(logs, events):
    annotated = logs.copy()
    for col in ["episode_severity", "episode_disturbance_type", "episode_name"]:
        annotated[col] = ""
    annotated["is_extreme_episode"] = False

    if events.empty:
        return annotated

    grouped_events = (
        events[events["event_type"].eq("disturbance")]
        .groupby(["experiment_run_id", "system_logs_id"], as_index=False)
        .agg(
            episode_severity=("severity_level", highest_severity),
            episode_disturbance_type=("disturbance_type", joined_unique),
            episode_name=("episode_name", joined_unique),
        )
        .sort_values(["experiment_run_id", "system_logs_id"])
    )

    for run_id, run_events in grouped_events.groupby("experiment_run_id"):
        run_events = run_events.sort_values("system_logs_id").reset_index(drop=True)
        for idx, event in run_events.iterrows():
            start_id = event["system_logs_id"]
            next_start = run_events.loc[idx + 1, "system_logs_id"] if idx + 1 < len(run_events) else np.inf
            mask = (
                annotated["experiment_run_id"].eq(run_id)
                & annotated["id"].ge(start_id)
                & annotated["id"].lt(next_start)
            )
            annotated.loc[mask, "episode_severity"] = event["episode_severity"]
            annotated.loc[mask, "episode_disturbance_type"] = event["episode_disturbance_type"]
            annotated.loc[mask, "episode_name"] = event["episode_name"]

    annotated["is_extreme_episode"] = annotated["episode_severity"].eq("extreme")
    return annotated

run_scenario_from_decisions = (
    system_logs.groupby("experiment_run_id")["decision"]
    .apply(infer_scenario_from_decisions)
    .to_dict()
)
run_scenario_from_events = infer_scenario_from_trial_events(trial_events)
run_scenario = run_scenario_from_decisions.copy()
run_scenario.update(run_scenario_from_events)

for run_id in system_logs["experiment_run_id"].unique():
    run_scenario.setdefault(run_id, "stable")

system_logs["scenario"] = system_logs["experiment_run_id"].map(run_scenario).fillna("unknown")
experiment_runs["scenario"] = experiment_runs["id"].map(run_scenario).fillna("stable")
system_logs = assign_disturbance_episode_annotations(system_logs, trial_events)

scenario_order = ["stable", "ph_low", "ph_high", "ec_low", "ec_high", "mixed"]
EXCLUDE_EXTREME_FROM_PRIMARY = True

def summarize_runs(df):
    summary = (
        df.groupby(["experiment_run_id", "control_strategy", "scenario"], as_index=False)
        .agg(
            readings=("id", "count"),
            first_timestamp=("timestamp", "min"),
            last_timestamp=("timestamp", "max"),
            ph_in_range_pct=("ph_within_range", lambda s: s.mean() * 100),
            ec_in_range_pct=("ec_within_range", lambda s: s.mean() * 100),
            both_in_range_pct=("both_within_range", lambda s: s.mean() * 100),
            dose_events=("is_dose_event", "sum"),
            wait_rows=("decision", lambda s: s.astype(str).str.startswith("wait").sum()),
            sensor_anomaly_rows=("decision", lambda s: (s == "sensor_anomaly").sum()),
        )
        .assign(dose_events_per_100=lambda df: df["dose_events"] / df["readings"] * 100)
    )
    for col in ["ph_in_range_pct", "ec_in_range_pct", "both_in_range_pct", "dose_events_per_100"]:
        summary[col] = summary[col].round(2)
    return summary

system_logs_full = system_logs.copy()
extreme_system_logs = system_logs_full[system_logs_full["is_extreme_episode"]].copy()
recoverable_system_logs = system_logs_full[~system_logs_full["is_extreme_episode"]].copy()
full_run_summary = summarize_runs(system_logs_full)

analysis_source = recoverable_system_logs.copy() if EXCLUDE_EXTREME_FROM_PRIMARY else system_logs_full.copy()

# Stable-range observations are first duration-capped, then row-balanced so neither
# controller receives extra no-disturbance readings in the primary benchmark.
STABLE_ALIGNMENT_HOURS = 3.0
analysis_mask = pd.Series(True, index=analysis_source.index)
stable_alignment_rows = []
for run_id, group in analysis_source[analysis_source["scenario"].eq("stable")].groupby("experiment_run_id"):
    start = group["timestamp"].min()
    cutoff = start + pd.Timedelta(hours=STABLE_ALIGNMENT_HOURS)
    retained_mask = group["timestamp"].le(cutoff)
    analysis_mask.loc[group.index] = retained_mask
    retained = group.loc[retained_mask]
    stable_alignment_rows.append({
        "experiment_run_id": run_id,
        "control_strategy": group["control_strategy"].iloc[0],
        "full_readings": len(group),
        "aligned_readings": len(retained),
        "window_start": start,
        "window_end": retained["timestamp"].max(),
        "hours_considered": (retained["timestamp"].max() - start).total_seconds() / 3600,
    })

stable_alignment = pd.DataFrame(stable_alignment_rows)
stable_alignment = stable_alignment.rename(columns={"aligned_readings": "time_capped_readings"})
min_stable_readings = int(stable_alignment["time_capped_readings"].min())
stable_alignment["row_balanced_readings"] = min_stable_readings
display(stable_alignment.round({"hours_considered": 2}))

time_aligned_logs = analysis_source.loc[analysis_mask].copy()
row_balance_mask = pd.Series(True, index=time_aligned_logs.index)
for run_id, group in time_aligned_logs[time_aligned_logs["scenario"].eq("stable")].groupby("experiment_run_id"):
    ranked = group.sort_values("timestamp")
    row_balance_mask.loc[ranked.index] = False
    row_balance_mask.loc[ranked.head(min_stable_readings).index] = True

system_logs = time_aligned_logs.loc[row_balance_mask].copy()
run_summary = summarize_runs(system_logs)
print(f"Primary Phase 2 analysis uses {len(system_logs):,} recoverable readings after excluding extreme disturbance episodes, capping stable windows at {STABLE_ALIGNMENT_HOURS:.1f} hours, and balancing stable readings to {min_stable_readings} per controller.")

# Current-workbook primary analysis audit.
actual_strategy_readings = system_logs["control_strategy"].value_counts().to_dict()
extreme_strategy_readings = extreme_system_logs["control_strategy"].value_counts().to_dict()
alignment_audit = pd.DataFrame([
    {
        "check": "primary recoverable readings: total",
        "observed": len(system_logs),
        "status": "current workbook",
    },
    *[
        {
            "check": f"primary recoverable readings: {strategy}",
            "observed": int(actual_strategy_readings.get(strategy, 0)),
            "status": "current workbook",
        }
        for strategy in ORDER
    ],
    {
        "check": "extreme stress-test readings: total",
        "observed": len(extreme_system_logs),
        "status": "reported separately",
    },
    *[
        {
            "check": f"extreme stress-test readings: {strategy}",
            "observed": int(extreme_strategy_readings.get(strategy, 0)),
            "status": "reported separately",
        }
        for strategy in ORDER
    ],
])
display(alignment_audit)
display(run_summary.sort_values(["scenario", "control_strategy", "experiment_run_id"]))


## 3. Trial Event Severity

The `trial_events` worksheet provides the primary event annotations for Phase 2, including disturbance annotations and Day 1 normal-operation annotations. Each event is linked to a specific `system_logs` row through `system_logs_id`, which makes the severity labels auditable against the actual pH and EC values recorded at the disturbance point. Severity-level summaries are used as supporting context for the completed benchmark. The headline comparison remains controller-level and scenario-level because recovery dynamics continue beyond the event rows.


In [ ]:
if trial_events.empty:
    trial_event_logs = pd.DataFrame()
    trial_event_summary = pd.DataFrame()
    severity_counts = pd.DataFrame()
    extreme_episode_summary = pd.DataFrame()
    print("No trial_events worksheet available. Severity summaries are unavailable.")
else:
    trial_event_logs = trial_events.merge(
        system_logs_full[[
            "id", "experiment_run_id", "control_strategy", "timestamp", "ph", "ec", "decision",
            "pump_activated", "dose_ml", "duration_ms", "ph_within_range", "ec_within_range",
            "both_within_range", "ph_deviation", "ec_deviation", "is_extreme_episode",
        ]],
        left_on="system_logs_id",
        right_on="id",
        how="left",
        suffixes=("_event", "_log"),
    )
    trial_event_logs["scenario"] = trial_event_logs["experiment_run_id_event"].map(run_scenario).fillna("unknown")

    def severity_label_matches_value(row):
        disturbance = row.get("disturbance_type", "")
        severity = row.get("severity_level", "")
        ph = row.get("ph", np.nan)
        ec = row.get("ec", np.nan)
        if disturbance == "ph_low":
            return {
                "mild": TARGET_PH_MIN - 0.20 <= ph < TARGET_PH_MIN,
                "moderate": TARGET_PH_MIN - 0.50 <= ph < TARGET_PH_MIN - 0.20,
                "severe": TARGET_PH_MIN - 0.80 <= ph < TARGET_PH_MIN - 0.50,
                "extreme": ph < TARGET_PH_MIN - 0.80,
            }.get(severity, False)
        if disturbance == "ph_high":
            return {
                "mild": TARGET_PH_MAX < ph <= TARGET_PH_MAX + 0.20,
                "moderate": TARGET_PH_MAX + 0.20 < ph <= TARGET_PH_MAX + 0.50,
                "severe": TARGET_PH_MAX + 0.50 < ph <= TARGET_PH_MAX + 0.80,
                "extreme": ph > TARGET_PH_MAX + 0.80,
            }.get(severity, False)
        if disturbance == "ec_low":
            return {
                "mild": TARGET_EC_MIN - 0.20 <= ec < TARGET_EC_MIN,
                "moderate": TARGET_EC_MIN - 0.50 <= ec < TARGET_EC_MIN - 0.20,
                "severe": TARGET_EC_MIN - 0.80 <= ec < TARGET_EC_MIN - 0.50,
                "extreme": ec < TARGET_EC_MIN - 0.80,
            }.get(severity, False)
        if disturbance == "ec_high":
            return {
                "mild": TARGET_EC_MAX < ec <= TARGET_EC_MAX + 0.20,
                "moderate": TARGET_EC_MAX + 0.20 < ec <= TARGET_EC_MAX + 0.50,
                "severe": TARGET_EC_MAX + 0.50 < ec <= TARGET_EC_MAX + 0.80,
                "extreme": ec > TARGET_EC_MAX + 0.80,
            }.get(severity, False)
        return False

    disturbance_event_logs = trial_event_logs[trial_event_logs["event_type"].eq("disturbance")].copy()
    normal_operation_event_logs = trial_event_logs[trial_event_logs["event_type"].eq("normal_operation")].copy()

    disturbance_event_logs["severity_label_matches_value"] = disturbance_event_logs.apply(severity_label_matches_value, axis=1)
    severity_from_episode = disturbance_event_logs["episode_name"].str.extract(r"_(mild|moderate|severe|extreme)(?:_|$)", expand=False)
    disturbance_event_logs["severity_label_matches_episode_name"] = severity_from_episode.isna() | severity_from_episode.eq(disturbance_event_logs["severity_level"])
    severity_validation = pd.DataFrame({
        "check": [
            "trial-event rows",
            "disturbance annotations",
            "normal-operation annotations",
            "disturbance severity label-to-reading mismatches",
            "disturbance severity label-to-episode-name mismatches",
        ],
        "count": [
            len(trial_event_logs),
            len(disturbance_event_logs),
            len(normal_operation_event_logs),
            int((~disturbance_event_logs["severity_label_matches_value"]).sum()),
            int((~disturbance_event_logs["severity_label_matches_episode_name"]).sum()),
        ],
    })

    trial_event_summary = (
        disturbance_event_logs.groupby(["controller_type", "disturbance_type", "severity_level"], as_index=False)
        .agg(
            events=("system_logs_id", "count"),
            mean_ph=("ph", "mean"),
            mean_ec=("ec", "mean"),
        )
    )

    severity_counts = (
        disturbance_event_logs.pivot_table(
            index=["controller_type", "disturbance_type"],
            columns="severity_level",
            values="system_logs_id",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["mild", "moderate", "severe", "extreme"], fill_value=0)
        .reset_index()
    )

    if extreme_system_logs.empty:
        extreme_episode_summary = pd.DataFrame()
    else:
        extreme_episode_summary = (
            extreme_system_logs.groupby(["control_strategy", "scenario"], as_index=False)
            .agg(
                readings=("id", "count"),
                ph_in_range_pct=("ph_within_range", lambda s: s.mean() * 100),
                ec_in_range_pct=("ec_within_range", lambda s: s.mean() * 100),
                both_in_range_pct=("both_within_range", lambda s: s.mean() * 100),
                dose_events=("is_dose_event", "sum"),
                dose_events_per_100=("is_dose_event", lambda s: s.mean() * 100),
                mean_ph_deviation=("ph_deviation", "mean"),
                mean_ec_deviation=("ec_deviation", "mean"),
            )
            .sort_values(["scenario", "control_strategy"])
        )

    severity_audit = severity_validation.copy()
    severity_audit["expected"] = [82, 80, 2, 0, 0]
    severity_audit["status"] = np.where(severity_audit["count"].eq(severity_audit["expected"]), "pass", "review")
    display(severity_audit)
    display(trial_event_summary.sort_values(["disturbance_type", "severity_level", "controller_type"]).round(3))
    display(severity_counts)
    display(extreme_episode_summary.round(2))


## 3. Statistical Utility Functions

The analysis uses binomial confidence intervals for proportions, two-proportion z tests for controller differences, Cohen's h for effect size, and cluster bootstrapping by experiment run. The cluster bootstrap is important because time-series readings are autocorrelated. It resamples whole runs instead of pretending every reading is independent.

In [ ]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
    return center - half, center + half


def two_prop_z_test(x1, n1, x2, n2):
    # H0: p1 == p2. Two-sided p-value using normal approximation.
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    if se == 0:
        return np.nan, np.nan
    z = (x1/n1 - x2/n2) / se
    p = erfc(abs(z) / np.sqrt(2))
    return z, p


def cohens_h(p1, p2):
    # Cohen's h for difference in proportions.
    p1 = np.clip(p1, 0, 1)
    p2 = np.clip(p2, 0, 1)
    return 2 * asin(sqrt(p1)) - 2 * asin(sqrt(p2))


def bootstrap_strategy_difference(df, metric_func, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    run_ids_by_strategy = {
        strategy: df.loc[df["control_strategy"].eq(strategy), "experiment_run_id"].dropna().unique()
        for strategy in ORDER
    }
    diffs = []
    for _ in range(n_boot):
        sampled_parts = {}
        for strategy in ORDER:
            run_ids = run_ids_by_strategy[strategy]
            sampled_ids = rng.choice(run_ids, size=len(run_ids), replace=True)
            sampled = pd.concat([df[(df["control_strategy"] == strategy) & (df["experiment_run_id"] == run_id)] for run_id in sampled_ids])
            sampled_parts[strategy] = sampled
        diffs.append(metric_func(sampled_parts["agentic_ai"]) - metric_func(sampled_parts["baseline"]))
    return np.percentile(diffs, [2.5, 50, 97.5])


def fmt_pct(x):
    return f"{x:.2f}%"

## 4. Overall Controller Benchmark

This section compares both controllers across the full Phase 2 export. The most important metric is **both pH and EC in range** because a hydroponic solution is only acceptable when both variables are within their target windows.

In [ ]:
def controller_summary(df):
    rows = []
    for strategy, group in df.groupby("control_strategy"):
        n = len(group)
        dose = int(group["is_dose_event"].sum())
        rows.append({
            "control_strategy": strategy,
            "readings": n,
            "ph_in_range_pct": group["ph_within_range"].mean() * 100,
            "ec_in_range_pct": group["ec_within_range"].mean() * 100,
            "both_in_range_pct": group["both_within_range"].mean() * 100,
            "avg_ph_deviation": group["ph_deviation"].mean(),
            "avg_ec_deviation": group["ec_deviation"].mean(),
            "p95_ph_deviation": group["ph_deviation"].quantile(0.95),
            "p95_ec_deviation": group["ec_deviation"].quantile(0.95),
            "ph_std": group["ph"].std(ddof=0),
            "ec_std": group["ec"].std(ddof=0),
            "dose_events": dose,
            "dose_events_per_100": dose / n * 100,
            "wait_rows": group["decision"].astype(str).str.startswith("wait").sum(),
            "sensor_anomaly_rows": group["decision"].eq("sensor_anomaly").sum(),
        })
    return pd.DataFrame(rows).sort_values("control_strategy")

overall = controller_summary(system_logs)
num_cols = overall.select_dtypes(include="number").columns
overall_display = overall.copy()
overall_display[num_cols] = overall_display[num_cols].round(4)
display(overall_display)

baseline = system_logs[system_logs["control_strategy"] == "baseline"]
agentic = system_logs[system_logs["control_strategy"] == "agentic_ai"]

comparison_rows = []
proportion_metrics = {
    "pH in range": "ph_within_range",
    "EC in range": "ec_within_range",
    "Both in range": "both_within_range",
    "Dose event rate": "is_dose_event",
}

for label, col in proportion_metrics.items():
    x_agent = int(agentic[col].sum())
    n_agent = len(agentic)
    x_base = int(baseline[col].sum())
    n_base = len(baseline)
    p_agent = x_agent / n_agent
    p_base = x_base / n_base
    z, pval = two_prop_z_test(x_agent, n_agent, x_base, n_base)
    ci_agent = wilson_ci(x_agent, n_agent)
    ci_base = wilson_ci(x_base, n_base)
    comparison_rows.append({
        "metric": label,
        "baseline_pct": p_base * 100,
        "baseline_95ci": f"[{ci_base[0]*100:.2f}, {ci_base[1]*100:.2f}]",
        "agentic_pct": p_agent * 100,
        "agentic_95ci": f"[{ci_agent[0]*100:.2f}, {ci_agent[1]*100:.2f}]",
        "agentic_minus_baseline_pp": (p_agent - p_base) * 100,
        "cohens_h": cohens_h(p_agent, p_base),
        "z_stat": z,
        "p_value": pval,
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({"baseline_pct":2, "agentic_pct":2, "agentic_minus_baseline_pp":2, "cohens_h":3, "z_stat":3, "p_value":5}))

In [ ]:
# Cluster-bootstrap intervals for headline differences.
bootstrap_metrics = {
    "Both in-range percentage-point difference": lambda g: g["both_within_range"].mean() * 100,
    "pH in-range percentage-point difference": lambda g: g["ph_within_range"].mean() * 100,
    "EC in-range percentage-point difference": lambda g: g["ec_within_range"].mean() * 100,
    "Dose-events-per-100 difference": lambda g: g["is_dose_event"].mean() * 100,
}

boot_rows = []
for label, func in bootstrap_metrics.items():
    lo, med, hi = bootstrap_strategy_difference(system_logs, func, n_boot=5000, seed=7)
    boot_rows.append({"metric": label, "bootstrap_2.5%": lo, "bootstrap_median": med, "bootstrap_97.5%": hi})

boot_summary = pd.DataFrame(boot_rows)
display(boot_summary.round(2))

### Results Interpretation

The Phase 2 dataset supports this main result:

> Agentic AI maintained comparable combined pH-and-EC range compliance while reducing normalized dosing activity.

The strongest result is dosing efficiency. Agentic AI used fewer dosing events, while p95 pH and EC deviations were higher than the baseline. Tail-deviation behavior is therefore reported as a residual risk.

The cluster-bootstrap interval is intentionally wider than the binomial confidence intervals because it resamples whole experiment runs rather than individual readings. This reflects the fact that Phase 2 runs were scenario-dependent time-series observations. Therefore, the bootstrap table should be interpreted as run-level uncertainty, while the aggregate table reports the observed controller-level performance.

## 5. Overall Performance Figures

In [ ]:
# Figure 1: Range compliance with Wilson CIs.
fig_data = []
for strategy, group in system_logs.groupby("control_strategy"):
    for label, col in [("pH", "ph_within_range"), ("EC", "ec_within_range"), ("Both", "both_within_range")]:
        x = int(group[col].sum())
        n = len(group)
        lo, hi = wilson_ci(x, n)
        fig_data.append({
            "Controller": display_controller(strategy),
            "Metric": label,
            "Percent": x/n*100,
            "CI_low": lo*100,
            "CI_high": hi*100,
        })
fig_df = pd.DataFrame(fig_data)
metric_order = ["pH", "EC", "Both"]
fig_df["Metric"] = pd.Categorical(fig_df["Metric"], categories=metric_order, ordered=True)
fig_df["Controller"] = pd.Categorical(fig_df["Controller"], categories=DISPLAY_ORDER, ordered=True)
fig_df = fig_df.sort_values(["Controller", "Metric"])

fig, ax = plt.subplots(figsize=(7.2, 4.2))
sns.barplot(data=fig_df, x="Metric", y="Percent", order=metric_order, hue="Controller", hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=ax)

# Add CI error bars aligned with Seaborn's hue-major patch order.
for patch, (_, row) in zip(ax.patches, fig_df.iterrows()):
    x = patch.get_x() + patch.get_width()/2
    y = row["Percent"]
    yerr_low = y - row["CI_low"]
    yerr_high = row["CI_high"] - y
    ax.errorbar(x, y, yerr=[[yerr_low], [yerr_high]], fmt="none", c="black", capsize=3, linewidth=1)

ax.set_title("Range Compliance by Controller")
ax.set_ylabel("Readings within target (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.legend(title="Controller", frameon=True, loc="upper right")
fig.tight_layout()
save_figure(fig, "fig1_range_compliance.png")
plt.show()


In [ ]:
# Figure 2: normalized dosing burden.
dose_fig = overall[["control_strategy", "dose_events_per_100", "dose_events"]].copy()
dose_fig["Controller"] = dose_fig["control_strategy"].map(display_controller)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
sns.barplot(data=dose_fig, x="Controller", y="dose_events_per_100", order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)
ax.set_title("Normalized Dosing Burden")
ax.set_xlabel("Controller")
ax.set_ylabel("Dosing events per 100 readings")
fig.tight_layout()
save_figure(fig, "fig2_dosing_burden.png")
plt.show()


## 6. Scenario-Level Results

Scenario-level analysis ties the statistics back to the seven-day experiment design. It helps explain *where* the improvement occurred: stable no-action behavior, pH correction, EC correction, or mixed disturbance handling.

In [ ]:
scenario_summary = (
    system_logs.groupby(["scenario", "control_strategy"], as_index=False)
    .agg(
        readings=("id", "count"),
        ph_in_range_pct=("ph_within_range", lambda s: s.mean()*100),
        ec_in_range_pct=("ec_within_range", lambda s: s.mean()*100),
        both_in_range_pct=("both_within_range", lambda s: s.mean()*100),
        dose_events=("is_dose_event", "sum"),
        dose_events_per_100=("is_dose_event", lambda s: s.mean()*100),
        wait_rows=("decision", lambda s: s.astype(str).str.startswith("wait").sum()),
        sensor_anomaly_rows=("decision", lambda s: (s == "sensor_anomaly").sum()),
    )
)
scenario_summary["scenario"] = pd.Categorical(scenario_summary["scenario"], categories=scenario_order, ordered=True)
scenario_summary = scenario_summary.sort_values(["scenario", "control_strategy"])

display(scenario_summary.round(2))

In [ ]:
# Figure 3: scenario-level combined compliance.
scenario_plot = scenario_summary.copy()
scenario_plot["Scenario"] = scenario_plot["scenario"].map(display_scenario)
scenario_plot["Controller"] = scenario_plot["control_strategy"].map(display_controller)
fig, ax = plt.subplots(figsize=(8.4, 4.4))
sns.barplot(
    data=scenario_plot,
    x="Scenario",
    y="both_in_range_pct",
    order=SCENARIO_DISPLAY_ORDER,
    hue="Controller",
    hue_order=DISPLAY_ORDER,
    palette=DISPLAY_PALETTE,
    ax=ax,
)
ax.set_title("Combined pH-and-EC Compliance by Scenario")
ax.set_xlabel("Scenario")
ax.set_ylabel("Both pH and EC within target (%)")
ax.set_ylim(0, 105)
ax.legend(title="Controller", frameon=True)
fig.tight_layout()
save_figure(fig, "fig3_scenario_compliance.png")
plt.show()


In [ ]:
# Figure 4: scenario-level dosing burden.
scenario_plot = scenario_summary.copy()
scenario_plot["Scenario"] = scenario_plot["scenario"].map(display_scenario)
scenario_plot["Controller"] = scenario_plot["control_strategy"].map(display_controller)
fig, ax = plt.subplots(figsize=(8.4, 4.4))
sns.barplot(
    data=scenario_plot,
    x="Scenario",
    y="dose_events_per_100",
    order=SCENARIO_DISPLAY_ORDER,
    hue="Controller",
    hue_order=DISPLAY_ORDER,
    palette=DISPLAY_PALETTE,
    ax=ax,
)
ax.set_title("Dosing Burden by Scenario")
ax.set_xlabel("Scenario")
ax.set_ylabel("Dosing events per 100 readings")
ax.legend(title="Controller", frameon=True)
fig.tight_layout()
save_figure(fig, "fig4_scenario_dosing_burden.png")
plt.show()


## 7. Pump-Level Actuation Analysis

Total dose volume must be interpreted by pump type. EC-down dosing uses dilution water and can be orders of magnitude larger than pH-up or pH-down dosing. Therefore, a single total-dose number can be misleading.

In [ ]:
dose_events = system_logs[system_logs["is_dose_event"]].copy()

pump_summary = (
    dose_events.groupby(["control_strategy", "pump_activated_norm"], as_index=False)
    .agg(
        events=("id", "count"),
        total_dose_ml=("dose_ml", "sum"),
        mean_dose_ml=("dose_ml", "mean"),
        max_dose_ml=("dose_ml", "max"),
        total_duration_s=("duration_ms", lambda s: s.sum()/1000),
    )
    .rename(columns={"pump_activated_norm": "pump_activated"})
)

display(pump_summary.round(2))

In [ ]:
# Figure 5: pump-specific event counts and dose volumes.
pump_summary_plot = pump_summary.copy()
pump_summary_plot["Pump"] = pump_summary_plot["pump_activated"].map(display_pump)
pump_summary_plot["Controller"] = pump_summary_plot["control_strategy"].map(display_controller)
pump_order = [label for pump, label in PUMP_LABELS.items() if pump in set(pump_summary_plot["pump_activated"])]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.barplot(data=pump_summary_plot, x="Pump", y="events", order=pump_order, hue="Controller", hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[0])
axes[0].set_title("Pump-Specific Dosing Events")
axes[0].set_xlabel("Pump")
axes[0].set_ylabel("Events")
axes[0].legend(title="Controller", frameon=True)

sns.barplot(data=pump_summary_plot, x="Pump", y="total_dose_ml", order=pump_order, hue="Controller", hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[1])
axes[1].set_title("Pump-Specific Total Dose Volume")
axes[1].set_xlabel("Pump")
axes[1].set_ylabel("Total dose (mL, log scale)")
axes[1].set_yscale("log")
axes[1].legend(title="Controller", frameon=True)

fig.tight_layout()
save_figure(fig, "fig5_pump_events_and_volume.png")
plt.show()


## 8. Response Time and Stability Checks

Response time is estimated from periodic logs. For each dosing event, the analysis searches forward to the first later reading where the targeted parameter returned to range.

The ten-reading stability estimate is dose-level: after each dose first returns the affected parameter to range, the next ten logged readings must remain in range. If drift occurs, that dose episode fails this strict check and any later re-dose starts a new response-and-confirmation episode.


In [ ]:
def target_for_pump(pump):
    if pump in ["ph_up", "ph_down"]:
        return "ph"
    if pump in ["ec_up", "ec_down"]:
        return "ec"
    return None

# Dose-level response estimate.
response_rows = []
for (strategy, run_id), group in system_logs.sort_values("timestamp").groupby(["control_strategy", "experiment_run_id"]):
    group = group.reset_index(drop=True)
    for idx, row in group[group["is_dose_event"]].iterrows():
        target = target_for_pump(row["pump_activated_norm"])
        if target is None:
            continue
        later = group.iloc[idx+1:].copy()
        resolved = later[later["ph_within_range"]] if target == "ph" else later[later["ec_within_range"]]
        if resolved.empty:
            response_rows.append({
                "control_strategy": strategy,
                "pump_activated": row["pump_activated_norm"],
                "target": target,
                "dose_log_id": row["id"],
                "response_minutes": np.nan,
                "ten_after_resolved_success": np.nan,
                "ten_after_resolved_available": 0,
            })
            continue

        first_resolved_index = resolved.index[0]
        response_minutes = (group.loc[first_resolved_index, "timestamp"] - row["timestamp"]).total_seconds() / 60

        # Stricter interpretation: ten confirmation readings AFTER first return-to-range.
        next10 = group.loc[first_resolved_index+1:first_resolved_index+10]
        if len(next10) == 10:
            ten_ok = bool(next10["ph_within_range"].all()) if target == "ph" else bool(next10["ec_within_range"].all())
        else:
            ten_ok = np.nan

        response_rows.append({
            "control_strategy": strategy,
            "pump_activated": row["pump_activated_norm"],
            "target": target,
            "dose_log_id": row["id"],
            "response_minutes": response_minutes,
            "ten_after_resolved_success": ten_ok,
            "ten_after_resolved_available": len(next10),
        })

responses = pd.DataFrame(response_rows)
response_summary = (
    responses.groupby("control_strategy", as_index=False)
    .agg(
        dose_events=("dose_log_id", "count"),
        resolved_events=("response_minutes", "count"),
        mean_response_min=("response_minutes", "mean"),
        median_response_min=("response_minutes", "median"),
        p75_response_min=("response_minutes", lambda s: s.dropna().quantile(0.75)),
        ten_after_resolved_n=("ten_after_resolved_success", "count"),
        ten_after_resolved_success_pct=("ten_after_resolved_success", lambda s: s.dropna().mean()*100),
    )
)

display(response_summary.round(2))


In [ ]:
# Figure 6: response-time distribution and post-dose stability estimate.
response_plot = responses.dropna(subset=["response_minutes"]).copy()
response_plot["Controller"] = response_plot["control_strategy"].map(display_controller)
stable_fig = response_summary.copy()
stable_fig["Controller"] = stable_fig["control_strategy"].map(display_controller)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.boxplot(data=response_plot, x="Controller", y="response_minutes", order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[0])
sns.stripplot(data=response_plot, x="Controller", y="response_minutes", order=DISPLAY_ORDER, color="black", alpha=0.35, size=3, ax=axes[0])
axes[0].set_title("Dose Response Time Distribution")
axes[0].set_xlabel("Controller")
axes[0].set_ylabel("Minutes to target range")

sns.barplot(data=stable_fig, x="Controller", y="ten_after_resolved_success_pct", order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[1])
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f%%", padding=3)
axes[1].set_title("Ten Readings After Return-to-Range")
axes[1].set_xlabel("Controller")
axes[1].set_ylabel("Success (%)")
axes[1].set_ylim(0, 105)

fig.tight_layout()
save_figure(fig, "fig6_response_and_stability.png")
plt.show()


## 9. Time-Series Figures

The pH and EC trajectories show how controller decisions relate to measured reservoir conditions over time.

In [ ]:
# Figure 7: pH and EC time series with dosing markers.
plot_logs = system_logs.sort_values("timestamp").copy()
plot_logs["time_index"] = np.arange(len(plot_logs))
plot_logs["Controller"] = plot_logs["control_strategy"].map(display_controller)

dose_points = plot_logs[plot_logs["is_dose_event"]]

fig, axes = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True)

sns.lineplot(data=plot_logs, x="time_index", y="ph", hue="Controller", hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[0], linewidth=1.3)
axes[0].axhspan(TARGET_PH_MIN, TARGET_PH_MAX, color="#54A24B", alpha=0.12, label="pH target")
axes[0].scatter(dose_points["time_index"], dose_points["ph"], marker="v", s=25, color="black", alpha=0.65, label="Dose event")
axes[0].set_title("pH Trajectory With Dosing Events")
axes[0].set_ylabel("pH")
axes[0].legend(loc="best", frameon=True)

sns.lineplot(data=plot_logs, x="time_index", y="ec", hue="Controller", hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=axes[1], linewidth=1.3, legend=False)
axes[1].axhspan(TARGET_EC_MIN, TARGET_EC_MAX, color="#54A24B", alpha=0.12, label="EC target")
axes[1].scatter(dose_points["time_index"], dose_points["ec"], marker="v", s=25, color="black", alpha=0.65, label="Dose event")
axes[1].set_title("EC Trajectory With Dosing Events")
axes[1].set_xlabel("Sequential log index")
axes[1].set_ylabel("EC (mS/cm)")
axes[1].legend(loc="best", frameon=True)

fig.tight_layout()
save_figure(fig, "fig7_timeseries_with_dosing.png")
plt.show()


## 10. Decision and Safety Behavior

This section supports discussion of *why* the controllers differ. The baseline acts directly from current readings, while Agentic AI adds confirmation, safety gate review, history-aware waiting, and anomaly detection.

In [ ]:
decision_counts = (
    system_logs.groupby(["control_strategy", "decision"])
    .size()
    .reset_index(name="count")
    .sort_values(["control_strategy", "count"], ascending=[True, False])
)
decision_counts["Controller"] = decision_counts["control_strategy"].map(display_controller)
decision_counts["Decision"] = decision_counts["decision"].map(display_decision)
display(decision_counts[["Controller", "Decision", "count"]])

decision_order = (
    decision_counts.groupby("Decision")["count"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=decision_counts, y="Decision", x="count", hue="Controller", order=decision_order, hue_order=DISPLAY_ORDER, palette=DISPLAY_PALETTE, ax=ax)
ax.set_title("Decision Distribution by Controller")
ax.set_xlabel("Rows")
ax.set_ylabel("Decision")
ax.legend(title="Controller", frameon=True)
fig.tight_layout()
save_figure(fig, "fig8_decision_distribution.png")
plt.show()


In [ ]:
# Extract Agentic AI metadata fields useful for discussion.
agentic_logs = system_logs[system_logs["control_strategy"] == "agentic_ai"].copy()
agentic_logs["agentic_action"] = agentic_logs["metadata"].apply(lambda m: m.get("agentic_action") if isinstance(m, dict) else None)
agentic_logs["reasoning_source"] = agentic_logs["metadata"].apply(lambda m: m.get("reasoning_source") if isinstance(m, dict) else None)
agentic_logs["safety_gate_source"] = agentic_logs["metadata"].apply(lambda m: m.get("safety_gate_source") if isinstance(m, dict) else None)
agentic_logs["dose_factor"] = agentic_logs["metadata"].apply(lambda m: m.get("applied_dose_adjustment_factor") or m.get("dose_adjustment_factor") if isinstance(m, dict) else np.nan)
agentic_logs["dose_factor"] = pd.to_numeric(agentic_logs["dose_factor"], errors="coerce")

agentic_metadata_summary = pd.DataFrame({
    "metric": [
        "Agentic rows",
        "Agentic dose rows",
        "Mean dose factor",
        "Minimum dose factor",
        "Maximum dose factor",
        "Deterministic safety gate rows",
    ],
    "value": [
        len(agentic_logs),
        int(agentic_logs["is_dose_event"].sum()),
        agentic_logs["dose_factor"].mean(),
        agentic_logs["dose_factor"].min(),
        agentic_logs["dose_factor"].max(),
        int(agentic_logs["safety_gate_source"].eq("deterministic_python").sum()),
    ],
})

display(agentic_metadata_summary.round(2))

## 11. Results Narrative

The following statements summarize the main statistical and operational results.

In [ ]:
base_row = overall.set_index("control_strategy").loc["baseline"]
agent_row = overall.set_index("control_strategy").loc["agentic_ai"]

both_gain = agent_row["both_in_range_pct"] - base_row["both_in_range_pct"]
ph_gain = agent_row["ph_in_range_pct"] - base_row["ph_in_range_pct"]
ec_gain = agent_row["ec_in_range_pct"] - base_row["ec_in_range_pct"]
dose_rate_reduction = base_row["dose_events_per_100"] - agent_row["dose_events_per_100"]
dose_count_reduction_pct = (base_row["dose_events"] - agent_row["dose_events"]) / base_row["dose_events"] * 100

agent_response = response_summary.set_index("control_strategy").loc["agentic_ai"]
base_response = response_summary.set_index("control_strategy").loc["baseline"]

print("Results Summary")
print("-" * 80)
print(f"Across {len(system_logs):,} primary recoverable Phase 2 readings, Agentic AI produced lower combined pH-and-EC in-range operation than baseline ({agent_row['both_in_range_pct']:.2f}% vs. {base_row['both_in_range_pct']:.2f}%, a difference of {both_gain:+.2f} percentage points).")
print(f"Agentic AI changed EC in-range operation by {ec_gain:+.2f} percentage points and pH in-range operation by {ph_gain:+.2f} percentage points relative to baseline.")
print(f"Normalized dosing activity decreased by {dose_rate_reduction:.2f} events per 100 readings, from {base_row['dose_events_per_100']:.2f} to {agent_row['dose_events_per_100']:.2f} events per 100 readings.")
print(f"Raw dosing events decreased from {int(base_row['dose_events'])} to {int(agent_row['dose_events'])}, a {dose_count_reduction_pct:.2f}% reduction.")
print(f"Median dose-response time was {agent_response['median_response_min']:.2f} min for Agentic AI and {base_response['median_response_min']:.2f} min for baseline. Mean response time is reported separately because EC-recovery episodes inflate the mean.")
print(f"The main caveat is that Agentic AI had higher p95 pH and EC deviations ({agent_row['p95_ph_deviation']:.3f}, {agent_row['p95_ec_deviation']:.3f}) than baseline ({base_row['p95_ph_deviation']:.3f}, {base_row['p95_ec_deviation']:.3f}), so tail deviations are reported as residual risk.")

## 12. Discussion Structure

The discussion is organized around these points:

1. **Overall control quality**: Agentic AI maintained comparable combined pH-and-EC range compliance.
2. **Dosing efficiency**: Agentic AI used fewer normalized dosing events, reducing pump activity and risk of overcorrection.
3. **Post-correction stability**: the ten-confirmation estimate is included as a strict dose-level supporting metric, not as the final trial-completion rule.
4. **EC-specific behavior**: EC correction remains the slowest and highest-volume control case because dilution and nutrient adjustment require mixing.
5. **Safety architecture**: Agentic AI decisions were bounded by deterministic safety gates, so the LLM/agent layer did not directly control hardware.
6. **Tradeoffs**: Agentic AI had higher p95 pH and EC deviations, so the result is lower actuation burden at similar aggregate compliance.
7. **Limitations**: the runs were operationally comparable but not perfectly randomized, response time was reconstructed from logs, and readings are autocorrelated.
8. **Future work**: paired trials, continuous probe traces, residual calibration analysis, and longer cultivation-cycle validation.
